**Experiment-6: GloVe Word Embeddings and Language Modelling**

Experiment-6.1

In [2]:
#without pretrained model
#import libraries

import numpy as np
import re
import json
from collections import Counter, defaultdict


#load the text corpus

file_path = "glove_corpus.txt"

with open("/content/glove_corpus.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Corpus loaded successfully!")
print("Total characters:", len(text))


#text preprocessing

text = text.lower()
text = re.sub(r"[^a-z\s]", " ", text)

words = text.split()

print("Total words:", len(words))
print("First 20 words:", words[:20])


#create vocabulary

word_counts = Counter(words)

vocab = sorted(word_counts.keys())

word_to_id = {
    word: index for index, word in enumerate(vocab)
}

id_to_word = {
    index: word for word, index in word_to_id.items()
}

vocab_size = len(vocab)

print("\nVocabulary Size:", vocab_size)


#create co-occurrence matrix (sparse)

window_size = 5

co_occurrence = defaultdict(float)

for i, word in enumerate(words):

    word_id = word_to_id[word]

    start = max(0, i - window_size)
    end = min(len(words), i + window_size + 1)

    for j in range(start, end):

        if i != j:

            context_word = words[j]
            context_id = word_to_id[context_word]

            distance = abs(i - j)

            #distance-weighted co-occurrence count
            co_occurrence[(word_id, context_id)] += 1.0 / distance


print("Co-occurrence pairs:", len(co_occurrence))


#prepare training data

pairs = list(co_occurrence.keys())

X_values = np.array(
    [co_occurrence[pair] for pair in pairs],
    dtype=np.float32
)

word_indices = np.array(
    [pair[0] for pair in pairs],
    dtype=np.int32
)

context_indices = np.array(
    [pair[1] for pair in pairs],
    dtype=np.int32
)

print("Training pairs:", len(X_values))


#initialize gloVe parameters

embedding_dim = 50
epochs = 25

learning_rate = 0.05
x_max = 100
alpha = 0.75

np.random.seed(42)

W = np.random.uniform(
    -0.5, 0.5, (vocab_size, embedding_dim)
).astype(np.float32)

W_context = np.random.uniform(
    -0.5, 0.5, (vocab_size, embedding_dim)
).astype(np.float32)

bias = np.zeros(vocab_size, dtype=np.float32)
bias_context = np.zeros(vocab_size, dtype=np.float32)


#initialize adaGrad accumulators

grad_W = np.ones_like(W)
grad_W_context = np.ones_like(W_context)

grad_bias = np.ones_like(bias)
grad_bias_context = np.ones_like(bias_context)


#train gloVe model

print("\nStarting GloVe Training...\n")

for epoch in range(epochs):

    total_loss = 0.0

    #shuffle training pairs
    order = np.random.permutation(len(X_values))

    for k in order:

        i = word_indices[k]
        j = context_indices[k]

        xij = X_values[k]

        # Weighting function
        weight = min((xij / x_max) ** alpha, 1.0)

        # Current vectors
        wi = W[i].copy()
        wj = W_context[j].copy()

        # Prediction
        prediction = (
            np.dot(wi, wj)
            + bias[i]
            + bias_context[j]
        )

        # Actual value
        target = np.log(xij)

        # Error
        error = prediction - target

        # Weighted loss
        loss = weight * (error ** 2)

        total_loss += 0.5 * loss

        # Gradients
        gradient = weight * error

        grad_wi = gradient * wj
        grad_wj = gradient * wi

        grad_bi = gradient
        grad_bj = gradient

        # AdaGrad updates for word vectors
        grad_W[i] += grad_wi ** 2

        W[i] -= (
            learning_rate * grad_wi /
            np.sqrt(grad_W[i])
        )

        # AdaGrad updates for context vectors
        grad_W_context[j] += grad_wj ** 2

        W_context[j] -= (
            learning_rate * grad_wj /
            np.sqrt(grad_W_context[j])
        )

        # Update biases
        grad_bias[i] += grad_bi ** 2

        bias[i] -= (
            learning_rate * grad_bi /
            np.sqrt(grad_bias[i])
        )

        grad_bias_context[j] += grad_bj ** 2

        bias_context[j] -= (
            learning_rate * grad_bj /
            np.sqrt(grad_bias_context[j])
        )

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"Loss: {total_loss:.4f}"
    )


#Generate Final Word Embeddings

embeddings = W + W_context

print("\nGloVe Training Completed!")
print("Embedding Shape:", embeddings.shape)


#Find Most Similar Words Using Cosine Similarity

def most_similar(target_word, top_n=10):

    target_word = target_word.lower()

    if target_word not in word_to_id:

        print(
            f"'{target_word}' is not present in the corpus vocabulary."
        )
        return

    target_index = word_to_id[target_word]

    target_vector = embeddings[target_index]

    # Normalize embeddings
    norms = np.linalg.norm(embeddings, axis=1)

    norms[norms == 0] = 1e-10

    normalized_embeddings = embeddings / norms[:, None]

    # Calculate cosine similarity
    similarities = np.dot(
        normalized_embeddings,
        normalized_embeddings[target_index]
    )

    # Exclude the target word itself
    similarities[target_index] = -np.inf

    # Get top similar words
    top_indices = np.argsort(similarities)[::-1][:top_n]

    print(
        f"\nTop {top_n} words similar to '{target_word}':\n"
    )

    for index in top_indices:

        print(
            f"{id_to_word[index]} : "
            f"{similarities[index]:.4f}"
        )


#Test the Model

most_similar("king", 10)


#Save Trained Embeddings

np.save("glove_embeddings.npy", embeddings)

with open("glove_vocabulary.json", "w") as file:
    json.dump(word_to_id, file)

print("\nEmbeddings saved successfully!")

Corpus loaded successfully!
Total characters: 800
Total words: 148
First 20 words: ['the', 'king', 'rules', 'the', 'kingdom', 'the', 'queen', 'rules', 'the', 'kingdom', 'the', 'king', 'is', 'a', 'powerful', 'man', 'the', 'queen', 'is', 'a']

Vocabulary Size: 68
Co-occurrence pairs: 779
Training pairs: 779

Starting GloVe Training...

Epoch 1/25 Loss: 12.1467
Epoch 2/25 Loss: 10.8904
Epoch 3/25 Loss: 9.9875
Epoch 4/25 Loss: 9.2911
Epoch 5/25 Loss: 8.7304
Epoch 6/25 Loss: 8.2651
Epoch 7/25 Loss: 7.8702
Epoch 8/25 Loss: 7.5283
Epoch 9/25 Loss: 7.2272
Epoch 10/25 Loss: 6.9579
Epoch 11/25 Loss: 6.7149
Epoch 12/25 Loss: 6.4933
Epoch 13/25 Loss: 6.2896
Epoch 14/25 Loss: 6.1004
Epoch 15/25 Loss: 5.9239
Epoch 16/25 Loss: 5.7583
Epoch 17/25 Loss: 5.6022
Epoch 18/25 Loss: 5.4544
Epoch 19/25 Loss: 5.3141
Epoch 20/25 Loss: 5.1805
Epoch 21/25 Loss: 5.0529
Epoch 22/25 Loss: 4.9311
Epoch 23/25 Loss: 4.8143
Epoch 24/25 Loss: 4.7022
Epoch 25/25 Loss: 4.5945

GloVe Training Completed!
Embedding Shape: (6

In [4]:
#with pretrained model

#Import Libraries
import gensim.downloader as api


#Load Your Corpus

file_path = "glove_corpus.txt"

with open("/content/glove_corpus.txt", "r", encoding="utf-8") as file:
    corpus = file.read()

print("Corpus loaded successfully!")
print("Total characters:", len(corpus))


#Load Pretrained GloVe Model

print("\nLoading Pretrained GloVe Model...")

model = api.load("glove-wiki-gigaword-100")

print("Pretrained GloVe Model Loaded Successfully!")


#Display Model Information

print("\nModel Information:")

print("Vocabulary Size:", len(model.key_to_index))

print("Embedding Dimensions:", model.vector_size)


#Check Whether King Exists

if "king" in model:

    print("\n'king' is present in the vocabulary.")

else:

    print("\n'king' is not present in the vocabulary.")


#Find Most Similar Words to King

similar_words = model.most_similar(
    "king",
    topn=10
)

print("\nTop 10 Most Similar Words to King:\n")

for word, similarity in similar_words:

    print(f"{word}: {similarity:.4f}")


#Display King Word Embedding

king_vector = model["king"]

print("\nWord Vector for King:")

print(king_vector)


#Calculate Similarity Between Two Words

similarity = model.similarity("king", "queen")

print(
    "\nSimilarity between King and Queen:",
    round(similarity, 4)
)


#Word Analogy

# King - Man + Woman = Queen

result = model.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5
)

print("\nWord Analogy: King - Man + Woman\n")

for word, score in result:

    print(f"{word}: {score:.4f}")


#Save King Vector

import numpy as np

np.save("pretrained_king_vector.npy", king_vector)

print("\nKing vector saved successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 54.4 MB/s  0:00:00
Corpus loaded successfully!
Total characters: 800

Loading Pretrained GloVe Model...
[==================================================] 100.0% 128.1/128.1MB downloaded
Pretrained GloVe Model Loaded Successfully!

Model Information:
Vocabulary Size: 400000
Embedding Dimensions: 100

'king' is present in the vocabulary.

Top 10 Most Similar Words to King:

prince: 0.7682
queen: 0.7508
son: 0.7021
brother: 0.6986
monarch: 0.6978
throne: 0.6920
kingdom: 0.6811
father: 0.6802
emperor: 0.6713
ii: 0.6676

Word Vector for King:
[-0.32307  -0.87616   0.21977   0.25268   0.22976   0.7388   -0.37954
 -0.35307  -0.84369  -1.1113   -0.30266   0.33178  -0.25113   0.30448
 -0.077491 -0.89815   0.092496 -1.1407   -0.58324   0.66869  -0.23122
 -0.95855   0.28262  -0.078848  0.75315   0.26584   0.3422   -0.33949
  0.95608   0.065641  0.45747   0.39835   0.57965   0.39267  -0.21851
  0.58795  -0.55999   0.63368  -0.043983 -0.68

Experiment-6.2

In [6]:

# Statistical Language Model Using N-gram Approach

# Import Libraries
import re
from collections import Counter


# Define Text Corpus
corpus = """
natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.
"""


# Text Preprocessing and Tokenization
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()


tokens = preprocess(corpus)

print("Preprocessed Tokens:")
print(tokens)

print("\nTotal Tokens:", len(tokens))


# Create Vocabulary
vocabulary = sorted(set(tokens))
vocab_size = len(vocabulary)

print("\nVocabulary:")
print(vocabulary)

print("\nVocabulary Size:", vocab_size)


# Generate N-Grams
def generate_ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


unigrams = generate_ngrams(tokens, 1)
bigrams = generate_ngrams(tokens, 2)
trigrams = generate_ngrams(tokens, 3)


# Calculate N-Gram Frequencies
unigram_counts = Counter(unigrams)
bigram_counts = Counter(bigrams)
trigram_counts = Counter(trigrams)

print("\nUnigram Frequencies:")
print(unigram_counts)

print("\nBigram Frequencies:")
print(bigram_counts)

print("\nTrigram Frequencies:")
print(trigram_counts)


# Calculate Bigram Probability
def bigram_probability(word1, word2):
    numerator = bigram_counts[(word1, word2)]
    denominator = unigram_counts[(word1,)]

    if denominator == 0:
        return 0

    return numerator / denominator


# Calculate Trigram Probability
def trigram_probability(word1, word2, word3):
    numerator = trigram_counts[(word1, word2, word3)]
    denominator = bigram_counts[(word1, word2)]

    if denominator == 0:
        return 0

    return numerator / denominator


print("\nBigram Probability:")
print(
    "P(language | natural) =",
    bigram_probability("natural", "language")
)

print("\nTrigram Probability:")
print(
    "P(processing | natural language) =",
    trigram_probability("natural", "language", "processing")
)


# Apply Laplace Smoothing
def smoothed_bigram_probability(word1, word2):
    numerator = bigram_counts[(word1, word2)] + 1
    denominator = unigram_counts[(word1,)] + vocab_size

    return numerator / denominator


def smoothed_trigram_probability(word1, word2, word3):
    numerator = trigram_counts[(word1, word2, word3)] + 1
    denominator = bigram_counts[(word1, word2)] + vocab_size

    return numerator / denominator


print("\nSmoothed Bigram Probability:")
print(
    smoothed_bigram_probability("natural", "language")
)


# Predict Next Word Using Bigram Model
def predict_bigram(previous_word, top_n=5):

    predictions = {}

    for word in vocabulary:
        probability = smoothed_bigram_probability(
            previous_word, word
        )

        predictions[word] = probability

    sorted_predictions = sorted(
        predictions.items(),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"\nBigram Predictions after '{previous_word}':")

    for word, probability in sorted_predictions[:top_n]:
        print(f"{word}: {probability:.4f}")


# Predict Next Word Using Trigram Model
def predict_trigram(word1, word2, top_n=5):

    predictions = {}

    for word in vocabulary:
        probability = smoothed_trigram_probability(
            word1, word2, word
        )

        predictions[word] = probability

    sorted_predictions = sorted(
        predictions.items(),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"\nTrigram Predictions after '{word1} {word2}':")

    for word, probability in sorted_predictions[:top_n]:
        print(f"{word}: {probability:.4f}")


# Test Bigram and Trigram Models
predict_bigram("natural")
predict_bigram("machine")

predict_trigram("natural", "language")
predict_trigram("machine", "learning")


# Display Model Summary
print("\n========== MODEL SUMMARY ==========")

print("Total Tokens:", len(tokens))
print("Vocabulary Size:", vocab_size)
print("Total Unigrams:", len(unigrams))
print("Total Bigrams:", len(bigrams))
print("Total Trigrams:", len(trigrams))

print("\nN-Gram Language Model Implemented Successfully!")


Preprocessed Tokens:
['natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence', 'natural', 'language', 'processing', 'enables', 'computers', 'to', 'understand', 'human', 'language', 'machine', 'learning', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence', 'deep', 'learning', 'is', 'a', 'part', 'of', 'machine', 'learning', 'language', 'models', 'learn', 'patterns', 'from', 'text', 'data', 'a', 'language', 'model', 'predicts', 'the', 'next', 'word', 'in', 'a', 'sentence', 'natural', 'language', 'processing', 'uses', 'language', 'models', 'artificial', 'intelligence', 'includes', 'machine', 'learning', 'and', 'deep', 'learning']

Total Tokens: 65

Vocabulary:
['a', 'and', 'artificial', 'branch', 'computers', 'data', 'deep', 'enables', 'from', 'human', 'in', 'includes', 'intelligence', 'is', 'language', 'learn', 'learning', 'machine', 'model', 'models', 'natural', 'next', 'of', 'part', 'patterns', 'predicts', 'processing', 'sentence', 'text', 'the'